# step C — RQ2 개입 (KV 치환으로 인과 확인, A2 생성)

**대응 RQ:** RQ2 — 매개 신호가 형태인가, 그리고 그것이 **인과**인가. (관측은 step B.)

**무엇을 하나** — 텍스트는 그대로 두고, 선행 위반 이름이 **L25에서 갖는 KV를 준수 값으로 치환**(내부 표현 수술)한 뒤
**준수 선호도**(모델이 지침대로 쓰려는 정도)가 회복되는지 본다.

**세 상태의 준수 선호 점수 `S = logP(준수 후보) − logP(위반 후보)`:**
- **S_깨끗(clean):** 선행에 위반이 없을 때 (기준 천장)
- **S_위반(baseline):** 위반 선행 그대로 (실패)
- **S_개입(intervened):** 위반이지만 L25만 준수 값으로 치환
- **회복률 = (S_개입 − S_위반) / (S_깨끗 − S_위반).**

**donor(공여) 3종:** `compliant`(같은 이름 준수판=주효과) / `unrelated_camel`(무관 준수형=형태 통제) / `unrelated_snake`(무관 위반형=음성 통제).

설계: `docs/stepC/plan.md`.

> **메모리(T4):** 개입은 output_attentions 안 씀(KV 캐시 편집), 짧은 프롬프트라 가볍다. eager 불필요.
> **검증:** 셀 5 요약에서 S_깨끗 > S_위반(위반이 실제로 준수 선호를 떨어뜨림)을 sanity check.
> **재개 가능:** 조건마다 저장, 이미 저장된 조건은 로드. GPU 없으면 매우 느림.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepC/KV-intervention
!git checkout stepC/KV-intervention
!git pull --quiet origin stepC/KV-intervention
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — donor 3종 × seed. 개입은 L25 Key+Value(방법 B), 선행은 전부 위반(n=0).
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
DONORS = ['compliant', 'unrelated_camel', 'unrelated_snake']
SEEDS = list(range(20))
LAYER = 25

def make(donor, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers=[LAYER], donor=donor),
        seed=s,
    )

conditions = [make(d, s) for d in DONORS for s in SEEDS]

PREDICTION = ('compliant 치환 시 준수 선호 회복(≈A2 79%). unrelated_camel(무관 준수형)도 '
              '회복하면 신호는 내용 아닌 형태. unrelated_snake(위반형)는 회복 안 됨.')
print(len(conditions), '조건 =', len(DONORS), 'donor x', len(SEEDS), 'seed  @L%d' % LAYER)

In [ ]:
# 실행 — 조건별 개입 + 즉시 저장(재개). 중간중간 donor별 누적 회복률 출력.
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL)   # 개입은 eager 불필요
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())

acc = defaultdict(list)
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step='stepC')
    if p.exists():
        rec = load_result(p); skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='stepC', rq='RQ2', prediction=PREDICTION))
        rec = load_result(p); new += 1
    r = rec.metrics.extra['recovery']; d = rec.condition.intervention.donor
    if r is not None:
        acc[d].append(r)
    if i % 15 == 0 or i == len(conditions):
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}')
        for d in DONORS:
            if acc[d]:
                print(f'    donor={d:<16} 회복률 평균 {sum(acc[d])/len(acc[d]):+.3f} (n={len(acc[d])})')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step='stepC')) for c in conditions]
print('로드:', len(records), '건 -> results/stepC/')

In [ ]:
# 요약 — donor별 회복률 + 세 상태 S. sanity: S_깨끗 > S_위반.
import pandas as pd, matplotlib.pyplot as plt

rows = [{'donor': r.condition.intervention.donor,
         'S_clean': r.metrics.extra['S_clean'], 'S_base': r.metrics.extra['S_base'],
         'S_int': r.metrics.extra['S_int'], 'recovery': r.metrics.extra['recovery'],
         'n_sub': r.metrics.extra['n_substituted_tokens']} for r in records]
df = pd.DataFrame(rows)

print('=== sanity: 위반이 준수 선호를 떨어뜨리는가 (S_clean > S_base?) ===')
print(df.groupby('donor')[['S_clean','S_base','S_int']].mean().round(3))
print()
print('=== donor별 회복률 ===')
print(df.groupby('donor')['recovery'].agg(['mean','std','count']).round(3))

# 플롯: donor별 회복률
g = df.groupby('donor')['recovery'].mean().reindex(DONORS)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
colors = ['#2E7D52', '#2563C9', '#C6771A']
ax[0].bar(range(len(g)), g.values, color=colors)
ax[0].axhline(0, color='#888', lw=.8); ax[0].axhline(1, color='#888', ls='--', lw=.8)
ax[0].set_xticks(range(len(g))); ax[0].set_xticklabels(g.index, rotation=12, fontsize=8)
ax[0].set_ylabel('recovery rate'); ax[0].set_title('Recovery by donor (1.0=clean)')
# 세 상태 S (donor=compliant)
sub = df[df.donor=='compliant'][['S_clean','S_base','S_int']].mean()
ax[1].bar(['clean','baseline','intervened'], sub.values, color=['#2E7D52','#B0392B','#2563C9'])
ax[1].set_ylabel('S (compliance preference)'); ax[1].set_title('Three states (donor=compliant)')
plt.tight_layout(); plt.show()

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('stepC_results', 'zip', 'results/stepC')
try:
    from google.colab import files
    files.download('stepC_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): stepC_results.zip', e)